In [1]:
%env AWS_PROFILE=platform-developer

env: AWS_PROFILE=platform-developer


In [2]:
import polars as pl
import pickle

CALM_RAW_WORKS_PATH = "/Users/brychtas/Documents/work-snapshots/calm-raw-works.parquet"
CALM_SOURCE_WORKS_PATH = "/Users/brychtas/Documents/work-snapshots/calm-source-works.parquet"
AXIELL_MARC_WORKS_PATH = "/Users/brychtas/Documents/work-snapshots/axiell-marc-works.parquet"
AXIELL_SOURCE_WORKS_PATH = "/Users/brychtas/Documents/work-snapshots/axiell-source-works.parquet"
AXIELL_RAW_WORKS_PATH = "/Users/brychtas/Documents/work-snapshots/axiell-raw-works.parquet"

def from_parquet_snapshot(path: str):
    df = pl.read_parquet(path)
    return {row["id"]: pickle.loads(row["body"]) for row in df.iter_rows(named=True)}

In [17]:
calm_raw_works = from_parquet_snapshot(CALM_RAW_WORKS_PATH)
print(f"CALM raw works count: {len(calm_raw_works)}")

CALM raw works count: 408085


In [20]:
axiell_marc_works = from_parquet_snapshot(AXIELL_MARC_WORKS_PATH)
print(f"Axiell MARC works count: {len(axiell_marc_works)}")

Axiell MARC works count: 216939


In [27]:
calm_source_works = from_parquet_snapshot(CALM_SOURCE_WORKS_PATH)
print(f"CALM source works count: {len(calm_source_works)}")

CALM source works count: 408085


In [25]:
axiell_source_works = from_parquet_snapshot(AXIELL_SOURCE_WORKS_PATH)
print(f"Axiell source works count: {len(axiell_source_works)}")

Axiell source works count: 211417


In [5]:
from adapters.extractors.oai_pmh.axiell.runtime import AXIELL_CONFIG

oai_client = AXIELL_CONFIG.build_oai_client()

def get_raw_axiell_work(collect_id: str):
    return oai_client.get_record(collect_id, metadata_prefix="oai_raw")

In [6]:
axiell_id_to_calm_id = {}
for work_id, work in axiell_source_works.items():
    if work.state.predecessor_identifier:
        calm_id = work.state.predecessor_identifier.value
        axiell_id_to_calm_id[work_id] = calm_id

calm_id_to_axiell_id = {}
for k, v in axiell_id_to_calm_id.items():
    calm_id_to_axiell_id[v] = k     

In [30]:
def standardise_works_permanent(calm_work, axiell_work):
    # Source identifiers will always be different
    del calm_work["state"]["sourceIdentifier"]
    del axiell_work["state"]["sourceIdentifier"]
        
    # Works no longer store siblings/children
    del calm_work["state"]["relations"]["siblingsPreceding"]
    del calm_work["state"]["relations"]["siblingsSucceeding"]
    del calm_work["state"]["relations"]["children"]

    # Source modified time will always be different
    del calm_work["state"]["sourceModifiedTime"]
    del axiell_work["state"]["sourceModifiedTime"]

    # Only Axiell works have a predecessor identifier
    del axiell_work["state"]["predecessorIdentifier"]

    # Only Axiell works have a modified time
    del axiell_work["state"]["modifiedTime"]

    # Versions will always be different
    del axiell_work["version"]
    del calm_work["version"]

    # Added automatically by the Elasticsearch ingest pipeline
    if "indexed_at" in calm_work:
        del calm_work["indexed_at"]

    # Replace CALM consecutive whitespaces with a single whitespace
    for i, note in enumerate(calm_work["data"]["notes"]):
        note["contents"] = re.sub(r'\s+', ' ', note["contents"]).strip()
    if "description" in calm_work["data"]:
        calm_work["data"]["description"] = re.sub(r'\s+', ' ', calm_work["data"]["description"]).strip()
    if "physicalDescription" in calm_work["data"]:
        calm_work["data"]["physicalDescription"] = re.sub(r'\s+', ' ', calm_work["data"]["physicalDescription"]).strip()

    # The Python pipeline uses the 'type' property more consistently than the Scala pipeline.
    for candidate in axiell_work["state"]["mergeCandidates"]:
        del candidate["id"]["type"]
    for subject in axiell_work["data"]["subjects"]:
        del subject["type"]
    for item in axiell_work["data"]["production"]:
        del item["dates"][0]["type"]
    
    return calm_work, axiell_work


def standardise_works(calm_work, axiell_work):
    """Normalise works before comparing"""

    calm_work, axiell_work = standardise_works_permanent(calm_work, axiell_work)

    for i, note in enumerate(axiell_work["data"]["notes"]):
        note["contents"] = note["contents"].replace(' rel="nofollow"', '')

    # Several Axiell works reference Sierra works not referenced by CALM in the source index.
    # I checked a few of them and they were all correct.
    calm_identifier_values = {i["value"] for i in calm_work["data"]["otherIdentifiers"]}
    for i, identifier in enumerate(axiell_work["data"]["otherIdentifiers"]):
        if identifier["identifierType"]["id"] == "sierra-system-number" and identifier["value"] not in calm_identifier_values:
            del axiell_work["data"]["otherIdentifiers"][i]

            for j, candidate in enumerate(axiell_work["state"]["mergeCandidates"]):
                if candidate["id"]["sourceIdentifier"]["value"] == identifier["value"]:
                    del axiell_work["state"]["mergeCandidates"][j]

    # Biographical notes are mapped to ownership notes: Known issue, documented in Notion.
    for i, note in enumerate(calm_work["data"]["notes"]):
        if note["noteType"]["id"] == "biographical-note":
            calm_work["data"]["notes"][i]["noteType"] = {"id": "ownership-note", "label": "Ownership note"}

    # Missing accession numbers: Known issue, documented in Notion.
    saw_accession_number = False
    for i, identifier in enumerate(calm_work["data"]["otherIdentifiers"]):
        if saw_accession_number:
            del calm_work["data"]["otherIdentifiers"][i]
        elif identifier["identifierType"]["id"] == "wellcome-accession-number":
            saw_accession_number = True

    # CALM pipeline validates CALM reference numbers and does not include a reference number as a merge candidate if it contains a hyphen.
    for i, candidate in enumerate(axiell_work["state"]["mergeCandidates"]):
        if candidate["id"]["sourceIdentifier"]["identifierType"]["id"] == "calm-ref-no" and "-" in candidate["id"]["sourceIdentifier"]["value"]:
            del axiell_work["state"]["mergeCandidates"][i]

    # Missing 'Closed' status in access condition, closed_until date used instead: Known issue, documented in Notion.
    if (calm_access_conditions := calm_work["data"]["items"][0]["locations"][0].get("accessConditions")):
        if calm_access_conditions[0]["status"]["type"] == "Closed":
            del calm_work["data"]["items"][0]["locations"][0]["accessConditions"]
            del axiell_work["data"]["items"][0]["locations"][0]["accessConditions"]
    if (axiell_access_conditions := axiell_work["data"]["items"][0]["locations"][0].get("accessConditions")):
        if axiell_access_conditions[0]["status"]["type"] == "Closed":
            del calm_work["data"]["items"][0]["locations"][0]["accessConditions"]
            del axiell_work["data"]["items"][0]["locations"][0]["accessConditions"]
        
    del axiell_work["data"]["production"]
    del calm_work["data"]["production"]

    return calm_work, axiell_work


def stream_normalised_source_works(limit: int | None = None):
    for i, (work_id, work) in enumerate(axiell_source_works.items()):
        if limit and i > limit:
            break

        if i % 10000 == 0 and i > 0:
            print(f"Streamed {i} works")
    
        if work.type == "Deleted" or not work.state.predecessor_identifier:
            continue      
    
        if work.state.predecessor_identifier:
            calm_id = work.state.predecessor_identifier.value
            calm_work = calm_source_works[calm_id]
    
            if calm_work["type"] in {"Deleted", "Invisible"}:
                continue

            calm_work, axiell_work = standardise_works(copy.deepcopy(calm_work), work.model_dump(exclude_none=True))
            yield {"id": calm_id, "work": calm_work}, {"id": work_id, "work": axiell_work}

In [71]:
from lxml import etree

axiell_id = "collect:110091580"
print(axiell_id_to_calm_id[axiell_id])
#print(axiell_marc_works[axiell_id])
print(calm_raw_works[axiell_id_to_calm_id[axiell_id]]["data"].get("AdminHistory"))

raw_record = get_raw_axiell_work(axiell_id).metadata
#print(etree.tostring(get_raw_axiell_work(axiell_id).metadata,  encoding="unicode"))

NS = "http://www.openarchives.org/OAI/2.0/"
print([el.text for el in raw_record.findall(f"{{{NS}}}object_history_note")])
#print([el.text for el in raw_record.findall(f"{{{NS}}}Dating/{{{NS}}}dating.date.start")])

# Do not ignore second production date!

d9f0d601-03b9-46ed-9c8b-3854feaac641
None
[]


In [29]:
axiell_id = "collect:110000025"
print(axiell_source_works[axiell_id].data.production)
print(calm_source_works[axiell_id_to_calm_id[axiell_id]]["data"].get("production"))

[ProductionEvent(label='1981', places=[], agents=[], dates=[Period(id=Unidentifiable(canonical_id=None, type='Unidentifiable'), label='1981', type='Period', range=DateTimeRange(from_time='1981-01-01T00:00:00Z', to_time='1981-12-31T23:59:59.999999999Z', label='1981'))], function=None)]
[{'label': '1981', 'places': [], 'agents': [], 'dates': [{'id': {'type': 'Unidentifiable'}, 'label': '1981', 'range': {'from': '1981-01-01T00:00:00Z', 'to': '1981-12-31T23:59:59.999999999Z', 'label': '1981'}}]}]


In [63]:
calm_id_to_axiell_id["f41de44d-64c5-4a0e-bb21-a003d89d8612"]

'collect:110163064'

In [107]:
# Accession number differences. Some CALM works have more than one accession number. Axiell works never have more than one.
accession_number_diffs = []

for calm, axiell in stream_normalised_source_works(10000):    
    calm_id, calm_work = calm["id"], calm["work"]
    
    axiell_id, axiell_work = axiell["id"], axiell["work"]

    if len([i for i in axiell_work["data"]["otherIdentifiers"] if i["identifierType"]["id"] == "wellcome-accession-number"]) > 1:
        print(axiell_id)

    diffs = DeepDiff(calm_work, axiell_work, ignore_order=True)
    if "iterable_item_removed" in diffs:
        if any("root['data']['otherIdentifiers']" in k for k in diffs["iterable_item_removed"]):
            diff_data = {"axiell_id": axiell_id, "calm_id": calm_id, "expected_values": []}
            print(work_id, calm_id)
            diff_works += 1
            for k in diffs["iterable_item_removed"]:
                if "root['data']['otherIdentifiers']" in k:
                    print(diffs["iterable_item_removed"][k])

collect:110211508 5eafae2c-b170-496f-892c-79297c4bf790
{'identifierType': {'id': 'wellcome-accession-number'}, 'ontologyType': 'Work', 'value': '2450'}
Streamed 10000 works


In [24]:
from lxml import etree

axiell_id = "collect:110000339"
print(calm_raw_works[axiell_id_to_calm_id[axiell_id]]['data'])

raw_axiell = get_raw_axiell_work(axiell_id)
print(etree.tostring(raw_axiell.metadata, pretty_print=True, encoding="unicode"))

{'Level': ['Series'], 'Location': [''], 'PreviousNumbers': [''], 'Created': ['22/01/2024'], 'ADMIN_DETAILS': [''], 'UserWrapped6': [''], 'CountryCode': ['GB'], 'UserText3': [''], 'Title': ['AIDS 2006'], 'ACCESS': [''], 'Arrangement': [''], 'CONTENT': [''], 'Transmission': ['Yes'], 'AccessStatus': [''], 'CreatorName': [''], 'Material': ['Archives - Hybrid'], 'Description': ['The 16th International AIDS Conference, Toronto; IAC XVI.'], 'Digitised': ['No'], 'Modifier': ['Sloyanv'], 'Copyright': [''], 'UserDate1': [''], 'ClosedUntil': [''], 'Creator': ['WELLCOME\\SymondsK'], 'Language': [''], 'IDENTITY': [''], 'RCN': ['R566873'], 'RelatedMaterial': [''], 'UserText5': ['215 Euston Road'], 'RefNo': ['PPCLI/B/3'], 'AccessConditions': [''], 'RepositoryCode': ['0120'], 'UserWrapped7': [''], 'CONTEXT': [''], 'ContentNote': [''], 'ALLIED_MATERIALS': [''], 'Date': ['2002-2006'], 'ConservationPriority': [''], 'Notes': [''], 'UserText6': [''], 'Extent': ['4 files, 5 digital items'], 'AltRefNo': ['PP

In [31]:
from deepdiff import DeepDiff
import re
import copy

# DICTIONARY ADDED
# SetOrdered(["root['data']['production'][0]['dates'][0]['range']"])

matching_works = 0
diff_works = 0

for calm, axiell in stream_normalised_source_works(1000):    
    calm_id, calm_work = calm["id"], calm["work"]
    axiell_id, axiell_work = axiell["id"], axiell["work"]

    diffs = DeepDiff(calm_work, axiell_work, ignore_order=True)
    if diffs:
        print(axiell_id, calm_id)
        diff_works += 1

        if "values_changed" in diffs:
            for key, val in diffs["values_changed"].items():
                print(key)
                print(f'\t {val["old_value"]}')
                print(f'\t {val["new_value"]}')
        if "iterable_item_removed" in diffs:
            print("REMOVED")
            print(diffs["iterable_item_removed"])
        if "iterable_item_added" in diffs:
            print("ADDED")
            print(diffs["iterable_item_added"])        
        if "dictionary_item_removed" in diffs:
            print("DICTIONARY REMOVED")
            print(diffs["dictionary_item_removed"])         
        if "dictionary_item_added" in diffs:
            print("DICTIONARY ADDED")
            print(diffs["dictionary_item_added"])                 

        print("------------------------------------")
    else:
        matching_works += 1

print(f"Matching works: {matching_works}")
print(f"Diff works: {diff_works}")

collect:110000073 6adee179-3907-4351-874e-b0ca91f11df9
ADDED
{"root['data']['items'][0]['locations'][0]['accessConditions'][0]": {'method': {'type': 'NotRequestable'}, 'status': {'type': 'Open'}}}
------------------------------------
collect:110000243 fb705de9-9a66-4af3-a48a-ee32b6214d34
root['data']['otherIdentifiers'][1]['value']
	 WA/HMM/CO/Sub/204-212
	 WA/HMM/CO/Sub/204
root['data']['collectionPath']['label']
	 WA/HMM/CO/Sub/204-212
	 WA/HMM/CO/Sub/204
root['data']['referenceNumber']
	 WA/HMM/CO/Sub/204-212
	 WA/HMM/CO/Sub/204
------------------------------------
collect:110000265 b9b217a5-7684-4b77-9f4f-5a5080fdf0fc
root['data']['workType']
	 Section
	 Series
------------------------------------
collect:110000339 e7dfd8fa-ff2b-41e8-b8b1-eebd904698e7
ADDED
{"root['data']['notes'][0]": {'noteType': {'id': 'terms-of-use', 'label': 'Terms of use'}, 'contents': 'This collection has been catalogued and is available to library members. Some items have access restrictions which are expla

In [13]:
deleted_calm_works = 0
deleted_axiell_works = 0
no_predecessor_id = 0

for work_id, work in axiell_source_works.items():
    if work.type == "Deleted":
        deleted_axiell_works += 1
        continue      
                

    if work.state.predecessor_identifier:
        calm_id = work.state.predecessor_identifier.value
        calm_work = calm_source_works[calm_id]

        if calm_work["type"] == "Deleted":
            deleted_calm_works += 1
            continue       
    else:
        no_predecessor_id += 1

print(f"Works deleted in Axiell: {deleted_axiell_works}")
print(f"Works deleted in CALM but visible in Axiell: {deleted_calm_works}")
print(f"Axiell works without a predecessor identifier: {no_predecessor_id}")

Works deleted in Axiell: 6686
Works deleted in CALM but visible in Axiell: 327
Axiell works without a predecessor identifier: 0


In [12]:
calm_raw_works["a881a9b5-ad2c-4b07-a512-9fa385ec2e1e"]

{'id': 'a881a9b5-ad2c-4b07-a512-9fa385ec2e1e',
 'data': {'Level': ['Section'],
  'Location': ['215;B11;MR;47;8;6'],
  'PreviousNumbers': ['MS.9292'],
  'UserWrapped5': ['These diaries were previously referenced as MS.9292 and changed to PP/SMA following an accrual to the collection in 2022.'],
  'Created': ['23/05/2023'],
  'UserWrapped6': [''],
  'CountryCode': ['GB'],
  'UserText3': [''],
  'Title': ['Personal and Biographical '],
  'ACCESS': [''],
  'Arrangement': [''],
  'CONTENT': [''],
  'Transmission': ['Yes'],
  'AccessStatus': ['Open'],
  'CreatorName': [''],
  'Material': ['Archives - Non-digital'],
  'Description': ["Diaries. The diaries cover the period from Stella Mason's childhood as the daughter of a progressive Headmaster at Buxton College, her early interest in drama, her enrolment at the Central School of Speech and Drama in London in 1938, her increasing interest and specialisation in speech therapy whilst at the School and the early days of her professional practice

In [14]:
for calm_id, calm_work in calm_source_works.items():
    if "data" not in calm_work:
        continue
    production = calm_work["data"]["production"]

    if not production:
        continue
        
    dates = production[0]['dates']
    production_label = production[0]["label"]
    dates_label = dates[0]['label']
    if 'range' not in dates[0]:
        continue
        axiell_id = calm_to_axiell[calm_id]
        print(production)
        print(transformed_axiell_works[axiell_id].data.production)
        break

    # Always identical to dates_label
    dates_range_label = dates[0]['range']['label']

    if production_label.removesuffix(".") != dates_label:
        print(production)
        break
        if calm_id not in calm_to_axiell:
            print("No Axiell work")
            continue
        axiell_production = transformed_axiell_works[calm_to_axiell[calm_id]].data.production
        axiell_production_label = axiell_production[0].label
        axiell_dates_label = axiell_production[0].dates[0].label
        print(production[0]["label"], "→", dates[0]['label'], " | ", axiell_production_label, "→", axiell_dates_label)
    

[{'label': '1958 1968', 'places': [], 'agents': [], 'dates': [{'id': {'type': 'Unidentifiable'}, 'label': '1958', 'range': {'from': '1958-01-01T00:00:00Z', 'to': '1958-12-31T23:59:59.999999999Z', 'label': '1958'}}, {'id': {'type': 'Unidentifiable'}, 'label': '1968', 'range': {'from': '1968-01-01T00:00:00Z', 'to': '1968-12-31T23:59:59.999999999Z', 'label': '1968'}}]}]


In [33]:
for work_id, record in marc_axiell_works.items():
    production_label = first_non_empty_subfield("264", "c", record)
    start_date = extract_production_start_date(record)
    end_date = extract_production_end_date(record)
    
    if (start_date is not None and end_date is None) or (start_date is None and end_date is not None):
        calm_id = axiell_to_calm.get(work_id)
        if calm_id:
            print(calm_source_works[calm_id]["data"]["production"])
        print(production_label, start_date, end_date)

[{'label': 'Aug 1939-Jul 12941', 'places': [], 'agents': [], 'dates': [{'id': {'type': 'Unidentifiable'}, 'label': 'Aug 1939-Jul 12941'}]}]
Aug 1939-Jul 12941 1939-08-01 None
2nd century - 21st century None 2099-12-31


In [79]:
import dateparser

dateparser.parse('c.1980s')

datetime.datetime(2026, 7, 1, 9, 49, 40, 127885)